In [2]:
import pandas as pd
import duckdb

data = [
    # Device A
    ["A", "2026-07-01 08:00:00", "NORMAL", 95],
    ["A", "2026-07-01 08:03:00", "ERROR", 90],
    ["A", "2026-07-01 08:05:00", "NORMAL", 90],
    ["A", "2026-07-01 08:08:00", "NORMAL", 80],

    # Device B
    ["B", "2026-07-01 08:01:00", "NORMAL", 88],
    ["B", "2026-07-01 08:04:00", "ERROR", 88],
    ["B", "2026-07-01 08:06:00", "NORMAL", 82],
    ["B", "2026-07-01 08:09:00", "NORMAL", 70],

    # Device C
    ["C", "2026-07-01 08:02:00", "ERROR", 100],
    ["C", "2026-07-01 08:07:00", "NORMAL", 92],
    ["C", "2026-07-01 08:10:00", "ERROR", 92],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "collect_time", "status", "temp_value"]
)

df["collect_time"] = pd.to_datetime(df["collect_time"])

print(df)



   device_id        collect_time  status  temp_value
0          A 2026-07-01 08:00:00  NORMAL          95
1          A 2026-07-01 08:03:00   ERROR          90
2          A 2026-07-01 08:05:00  NORMAL          90
3          A 2026-07-01 08:08:00  NORMAL          80
4          B 2026-07-01 08:01:00  NORMAL          88
5          B 2026-07-01 08:04:00   ERROR          88
6          B 2026-07-01 08:06:00  NORMAL          82
7          B 2026-07-01 08:09:00  NORMAL          70
8          C 2026-07-01 08:02:00   ERROR         100
9          C 2026-07-01 08:07:00  NORMAL          92
10         C 2026-07-01 08:10:00   ERROR          92


# 二、题目要求

## 分别使用 SQL 和 Pandas 完成。

### Part 1：生成三种排名

* **按照每个设备内部的 temp_value 从高到低排名，生成三列：**

- `row_number_rank`
- `rank_value`
- `dense_rank_value`

**最终字段：**

- `device_id`
- `collect_time`
- `status`
- `temp_value`
- `row_number_rank`
- `rank_value`
- `dense_rank_value`

**SQL 要求使用：**

- `ROW_NUMBER()`
- `RANK()`
- `DENSE_RANK()`

**Pandas 要求使用：**

- `sort_values()`
- `groupby().cumcount()`
- `rank(method=...)`

# 三、预期结果

* **结果大致应该是：**

device_id | collect_time          | status | temp_value | row_number_rank | rank_value | dense_rank_value
----------|-----------------------|--------|------------|-----------------|------------|-----------------
A         | 2026-07-01 08:00:00   | NORMAL | 95         | 1               | 1          | 1
A         | 2026-07-01 08:05:00   | NORMAL | 90         | 2               | 2          | 2
A         | 2026-07-01 08:03:00   | ERROR  | 90         | 3               | 2          | 2
A         | 2026-07-01 08:08:00   | NORMAL | 80         | 4               | 4          | 3
B         | 2026-07-01 08:04:00   | ERROR  | 88         | 1               | 1          | 1
B         | 2026-07-01 08:01:00   | NORMAL | 88         | 2               | 1          | 1
B         | 2026-07-01 08:06:00   | NORMAL | 82         | 3               | 3          | 2
B         | 2026-07-01 08:09:00   | NORMAL | 70         | 4               | 4          | 3
C         | 2026-07-01 08:02:00   | ERROR  | 100        | 1               | 1          | 1
C         | 2026-07-01 08:10:00   | ERROR  | 92         | 2               | 2          | 2
C         | 2026-07-01 08:07:00   | NORMAL | 92         | 3               | 2          | 2

* **重点观察：**

**A 设备：**
- `95 → rank 1`
- `90 → rank 2`
- `90 → rank 2`
- `80 → rank 4`

所以 RANK 会跳过 3。

但 `DENSE_RANK` 是：
- `95 → 1`
- `90 → 2`
- `90 → 2`
- `80 → 3`

所以 DENSE_RANK 不跳号。

# 四、Part 2：筛选温度前两档

* **然后再做一个筛选：**

- 找出每个设备中，温度排名处于前 2 档的记录。

- 注意，这里说的是“前 2 档温度”，不是“前 2 条记录”。

- 所以应该使用：

`DENSE_RANK() <= 2`

而不是：

`ROW_NUMBER() <= 2`

也不能：

`RANK() <= 2`

因为：

`RANK()` 会跳号，如果第一名有两个，那么第2名的名次就会跳到3，用小于等于2来筛选，会把第2名踢掉。

预期结果应该包含：

- `A：95、90、90`
- `B：88、88、82`
- `C：100、92、92`



In [13]:
# SQL轨道

query = """
WITH rank_table AS (
SELECT
    device_id,
    collect_time,
    status,
    temp_value,
    ROW_NUMBER() OVER(PARTITION BY device_id ORDER BY temp_value DESC,collect_time ASC) AS row_number_rank,
    RANK()  OVER(PARTITION BY device_id ORDER BY temp_value DESC) AS rank_value,
    DENSE_RANK()  OVER(PARTITION BY device_id ORDER BY temp_value DESC) AS dense_rank_value
FROM df
),
top_2_temp AS(
    SELECT
        device_id,
    collect_time,
    status,
    temp_value,
    row_number_rank,
    rank_value,
    dense_rank_value
FROM rank_table
WHERE dense_rank_value <= 2
)
SELECT *
FROM top_2_temp
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,collect_time,status,temp_value,row_number_rank,rank_value,dense_rank_value
0,C,2026-07-01 08:02:00,ERROR,100,1,1,1
1,C,2026-07-01 08:07:00,NORMAL,92,2,2,2
2,C,2026-07-01 08:10:00,ERROR,92,3,2,2
3,B,2026-07-01 08:01:00,NORMAL,88,1,1,1
4,B,2026-07-01 08:04:00,ERROR,88,2,1,1
5,B,2026-07-01 08:06:00,NORMAL,82,3,3,2
6,A,2026-07-01 08:00:00,NORMAL,95,1,1,1
7,A,2026-07-01 08:03:00,ERROR,90,2,2,2
8,A,2026-07-01 08:05:00,NORMAL,90,3,2,2


In [ ]:
# PANDAS轨道

df_pd = (
    df
    .sort_values(by=['device_id','temp_value','collect_time'],ascending=[True,False,True])
    .assign(
        row_number_rank = lambda x:x.groupby('device_id').cumcount() + 1,
        rank_value = lambda x:(
            x.groupby('device_id')['temp_value']
            .rank(method='min',ascending=False)
            .astype(int)
        ),
        dense_rank_value = lambda x:(
            x.groupby('device_id')['temp_value']
            .rank(method='dense',ascending=False)
            .astype(int)
        )
    )
    .loc[lambda x:x['dense_rank_value'] <= 2]
    .reset_index(drop=True)
)
df_pd

,device_id,collect_time,status,temp_value,row_number_rank,rank_value,dense_rank_value
0,A,2026-07-01 08:00:00,NORMAL,95,1,1,1
1,A,2026-07-01 08:03:00,ERROR,90,2,2,2
2,A,2026-07-01 08:05:00,NORMAL,90,3,2,2
4,B,2026-07-01 08:01:00,NORMAL,88,1,1,1
5,B,2026-07-01 08:04:00,ERROR,88,2,1,1
6,B,2026-07-01 08:06:00,NORMAL,82,3,3,2
8,C,2026-07-01 08:02:00,ERROR,100,1,1,1
9,C,2026-07-01 08:07:00,NORMAL,92,2,2,2
10,C,2026-07-01 08:10:00,ERROR,92,3,2,2
